In [1]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn

DATA_DIR = "IRMAS-TrainingData/IRMAS-TrainingData"

In [2]:
X, y = [], []

for label in os.listdir(DATA_DIR):
    class_dir = os.path.join(DATA_DIR, label)
    if not os.path.isdir(class_dir):
        continue

    for fname in os.listdir(class_dir):
        if not fname.lower().endswith(".wav"):
            continue
        audio, sr = librosa.load(os.path.join(class_dir, fname))
        mel = librosa.feature.melspectrogram(y=audio, sr=sr)
        X.append(librosa.power_to_db(mel))
        y.append(label)

X = np.array(X) 
y = np.array(y)

c:\Users\Wreck\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
np.savez_compressed("dataset.npz", X=X, y=y)

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.dropout_conv1 = nn.Dropout2d(0.1)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.dropout_conv2 = nn.Dropout2d(0.1)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.dropout_conv3 = nn.Dropout2d(0.1)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.dropout_conv4 = nn.Dropout2d(0.1)
        self.pool4 = nn.MaxPool2d(2, 2)

        self.relu = nn.ReLU()
        self.global_pool = nn.AdaptiveAvgPool2d((8, 2))
        
        self.fc1 = nn.Linear(128 * 8 * 2, 128)
        self.dropout_fc = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout_conv1(x)
        x = self.pool1(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout_conv2(x)
        x = self.pool2(x)
        
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout_conv3(x)
        x = self.pool3(x)
        
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu(x)
        x = self.dropout_conv4(x)
        x = self.pool4(x)
        
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout_fc(x)
        x = self.fc2(x)
        return x

In [ ]:
import torchvision.models as models
import torch.nn as nn

class InstrumentClassifierResNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        
        self.resnet = models.resnet18(pretrained=True)
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        for param in self.resnet.layer1.parameters():
            param.requires_grad = False
        for param in self.resnet.layer2.parameters():
            param.requires_grad = False
        
        for param in self.resnet.layer3.parameters():
            param.requires_grad = True
        for param in self.resnet.layer4.parameters():
            param.requires_grad = True
        
        for param in self.resnet.conv1.parameters():
            param.requires_grad = True
        for param in self.resnet.bn1.parameters():
            param.requires_grad = True
        
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, n_classes)
        )
    
    def forward(self, x):
        return self.resnet(x)



In [ ]:
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import functional as F
import torchaudio.transforms as T

device = (
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)

class SpecAugment:
    def __init__(self):
        self.freq_mask = T.FrequencyMasking(freq_mask_param=60)   #60
        self.time_mask = T.TimeMasking(time_mask_param=80)       #80 = 94-82
    
    def __call__(self, spec):
        if torch.rand(1) > 0.5:
            spec = self.freq_mask(spec)
        if torch.rand(1) > 0.5:
            spec = self.time_mask(spec)
        return spec


data = np.load("dataset.npz")
X, y = data["X"], data["y"]

le = LabelEncoder()
y_int = le.fit_transform(y)
n_classes = len(le.classes_)

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # add channel dim
y_tensor = torch.tensor(y_int, dtype=torch.long)

# 80/20 train/val split, stratified so class proportions stay consistent
X_train, X_val, y_train, y_val = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42, stratify=y_tensor
)

# Build datasets + loaders
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

augment = SpecAugment()


print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Classes:", le.classes_)

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt
from IPython.display import clear_output

EPOCHS = 75

model = InstrumentClassifierResNet(n_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

lossFn = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

train_accs = []
val_accs = []

maxAcc = 0.0
for epoch in range(EPOCHS):
    model.train()
    trainLoss = 0.0
    correct = 0
    total = 0

    for inputs, targets in train_loader:
        inputs = augment(inputs)
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model(inputs)
        loss = lossFn(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        trainLoss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    avgTrainLoss = trainLoss / len(train_loader)
    trainAcc = correct / total

    model.eval()
    valLoss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = lossFn(outputs, targets)

            valLoss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)

    avgValLoss = valLoss / len(val_loader)
    valAcc = correct / total
    scheduler.step(avgValLoss)

    train_accs.append(trainAcc)
    val_accs.append(valAcc)

    clear_output(wait=True)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {avgTrainLoss:.4f}, Train Acc: {trainAcc:.4f} | "
          f"Val Loss: {avgValLoss:.4f}, Val Acc: {valAcc:.4f}")

    if valAcc > maxAcc:
        maxAcc = valAcc
        torch.save(model.state_dict(), "model.pth")
    plt.figure(figsize=(8, 4))
    plt.plot(train_accs, label="Train Accuracy")
    plt.plot(val_accs, label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.show()

    print(f"Max Accuracy: {maxAcc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

model = InstrumentClassifierResNet(n_classes=11).to(device)
model.load_state_dict(torch.load('model.pth', map_location=device))
model.eval()

all_preds = []
all_targets = []

print("Generating predictions...")
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

# Create confusion matrix
cm = confusion_matrix(all_targets, all_preds)

# Plot with seaborn
plt.figure(figsize=(14, 12))
sns.heatmap(
    cm, 
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    cbar_kws={'label': 'Count'},
    linewidths=0.5
)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Instrument Classification', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Print per-class accuracy
print("\nPer-class Validation Accuracy:")
print("-" * 30)
class_accuracies = cm.diagonal() / cm.sum(axis=1)
for i, class_name in enumerate(le.classes_):
    acc = class_accuracies[i]
    print(f"{class_name:8s}: {acc:.2%}")

print("-" * 30)
print(f"Overall:  {class_accuracies.mean():.2%}")